<a href="https://colab.research.google.com/github/gumeeee/open-source-genai-huggingface/blob/main/week-3-day-5-meeting-minutes-product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


## Again - please note: pro-tip for using Colab:

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00


In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [3]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# New capability - connect this Colab to your Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3
# Place the file on your drive in a folder called llms, and call it denver_extract.mp3

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [ ]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

# STEP 1: Transcribe Audio

## Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

In [ ]:
open_source_transcription = transcription

## Option 2: Use OpenAI for Transcription

In [ ]:
# Sign in to OpenAI using Secrets in Colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# STEP 2: Analyze & Report

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))

# Student contribution

Student Emad S. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D

In [ ]:
1. 🎯 TL;DR (em 3 linhas)
Este projeto automatiza a criação de atas de reuniões transformando áudio em texto (via Whisper ou GPT-4o) e processando essa transcrição com o Llama-3.2 localmente via quantização 4-bit. O objetivo é extrair resumos, pontos de discussão e planos de ação estruturados de forma eficiente.

2. 📌 Resumo Executivo
O notebook demonstra um pipeline completo de Engenharia de Prompt e LLMs aplicados a produtividade. Ele aborda desde a ingestão de áudio (usando Google Drive) até a transcrição (comparando modelos Open Source vs. API proprietária). O diferencial técnico é a implementação do modelo Llama-3.2-3B com quantização bitsandbytes, permitindo que um modelo de linguagem poderoso rode em hardware acessível (como a GPU T4 do Colab) para analisar textos longos e gerar documentos estruturados.

3. 🧠 Conceitos-Chave
Transcrição (ASR): Conversão de fala em texto. No código, usa-se o Whisper (Open Source) e a API da OpenAI.
Quantização (4-bit/NF4): Técnica para reduzir o tamanho do modelo (ex: Llama-3.2) para que ele caiba na memória da GPU sem perder muita precisão.
System Message: Instrução mestre que define o 'comportamento' e tom do modelo (ex: 'Você produz atas...').
Chat Template: Formatação específica exigida pelo Llama para entender quem é o usuário e quem é o sistema.
Streaming (TextStreamer): Exibição das palavras à medida que são geradas, melhorando a experiência do usuário.
4. 🔍 Aprofundamento
A grande sacada aqui é o Trade-off entre Custo e Privacidade/Controle. Ao usar o Whisper local e o Llama quantizado, você reduz custos de API e mantém os dados processados no seu ambiente. O uso de BitsAndBytesConfig (NF4) é crucial: ele não apenas comprime o modelo, mas usa o tipo de dado bfloat16 para cálculos, o que equilibra velocidade e estabilidade numérica. Um ponto crítico é que transcrições brutas costumam ser 'sujas' (com vícios de linguagem); o prompt do sistema precisa ser robusto o suficiente para filtrar o ruído e focar apenas no que gera valor (decisões e donos de tarefas).

5. 💡 Exemplos Práticos
Cotidiano: Transformar um áudio de 10 minutos de uma reunião de condomínio em uma lista de tarefas no WhatsApp.
Negócios: Analisar chamadas de vendas para identificar automaticamente as principais objeções dos clientes.
Exemplo com Código Real:
# Trecho do notebook que configura a compressão do modelo
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)
Projetos Grandes: Integração em sistemas de CRM (como Salesforce) onde cada gravação de reunião é automaticamente anexada como um resumo estruturado no perfil do cliente.
6. 🔗 Conexões e Contexto
Este tema se conecta com Processamento de Linguagem Natural (NLP) e HCI (Interação Humano-Computador). A origem vem dos modelos Encoder-Decoder (como o Whisper). Autores como Andrej Karpathy e empresas como Meta (Llama) e Hugging Face são os pilares que tornam esse notebook possível.

7. ⚙️ Aplicação Prática
Uso prático: Automatizar a documentação de reuniões semanais (Sprints).
Decisões: Escolher entre API (rápido, pago) ou Local (privacidade, configuração técnica).
Passos Concretos: 1. Subir seu áudio no Drive; 2. Rodar a transcrição Whisper; 3. Ajustar o system_message para o formato de ata da sua empresa.
8. ❓ Perguntas Socráticas
E se o áudio tiver 3 pessoas falando ao mesmo tempo, como a transcrição mudaria?
Por que usamos quantização 4-bit em vez de carregar o modelo completo?
Como você explicaria para uma criança que o computador 'entende' o que foi falado na reunião?
O que acontece com a qualidade da ata se o prompt do usuário for curto demais?
Se o Llama gerar uma informação que não estava no áudio (alucinação), como detectaríamos?
9. ⚠️ Armadilhas Comuns
Esquecer o T4: Tentar rodar o Llama sem ativar a GPU no Colab.
Caminho do Arquivo: Errar o local do áudio no Google Drive (/content/drive/MyDrive/...).
Limites de Tokens: Transcrições gigantes podem estourar a janela de contexto do Llama-3.2.
10. 📚 Para Aprofundar Mais
Livros: Natural Language Processing with Transformers (Lewis Tunstall).
Vídeos: Documentação da Meta sobre o Llama 3.2.
Pesquisa: Como melhorar a 'Diarização' (identificar quem fala o quê) em reuniões com múltiplos participantes.